# Notebook 00: Project Setup and FastF1 Sanity Check

The goal of this notebook is the following:

1. Set up folder paths and FastF1 cache
2. Load one session successfully
3. Inspect key tables (results + laps)
4. Save sample data to disk


## 0) One-time installs

Make sure to install the dependencies by installing requirements.txt

```
pip install -r requirements.txt
```


In [1]:
# Core Python and data libraries
from pathlib import Path
import pandas as pd
import numpy as np

# FastF1 imports
import fastf1

print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
# Resolve project root whether notebook runs from repo root or notebooks folder
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

data_dir = project_root / 'data'
raw_dir = data_dir / 'raw'
processed_dir = data_dir / 'processed'
cache_dir = raw_dir / 'fastf1_cache'

for folder in [data_dir, raw_dir, processed_dir, cache_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# Enable local cache so repeated API calls are much faster
fastf1.Cache.enable_cache(str(cache_dir))

print('Project root:', project_root)
print('FastF1 cache:', cache_dir)

Project root: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction
FastF1 cache: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\fastf1_cache


## 1) Load one real session (sanity check)

Use a recent event so data is available and easier to inspect.

You can change these values later.


In [4]:
season = 2025
gp_name = 'Bahrain'
session_type = 'Q'  # Q=Qualifying, R=Race, FP1/FP2/FP3=Practice

session = fastf1.get_session(season, gp_name, session_type)

# laps=True loads lap-level data. telemetry=False keeps this first run faster.
session.load(laps=True, telemetry=False, weather=True, messages=False)

print('Loaded session:', session.event['EventName'], session_type)
print('Session date:', session.date)
print('Drivers found:', len(session.drivers))

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['81', '63', '16', '12', '10', '4', '1', '55', '44', '22', '7', '6', '14', '31', '23', '27', '30', '5', '18', '87']


Loaded session: Bahrain Grand Prix Q
Session date: 2025-04-12 16:00:00
Drivers found: 20


In [6]:
# inspect classification/result table
results_df = session.results.copy()

candidate_cols = [
    'Abbreviation', 'FullName', 'TeamName', 'Position',
    'Q1', 'Q2', 'Q3', 'GridPosition', 'Status', 'Points'
]
existing_cols = [c for c in candidate_cols if c in results_df.columns]

display(results_df[existing_cols].head(20))
print('Results shape:', results_df.shape)

,Abbreviation,FullName,TeamName,Position,Q1,Q2,Q3,GridPosition,Status,Points
81,PIA,Oscar Piastri,McLaren,1.0,0 days 00:01:31.392000,0 days 00:01:30.454000,0 days 00:01:29.841000,NaN,,NaN
63,RUS,George Russell,Mercedes,2.0,0 days 00:01:31.494000,0 days 00:01:30.664000,0 days 00:01:30.009000,NaN,,NaN
16,LEC,Charles Leclerc,Ferrari,3.0,0 days 00:01:31.454000,0 days 00:01:30.724000,0 days 00:01:30.175000,NaN,,NaN
12,ANT,Kimi Antonelli,Mercedes,4.0,0 days 00:01:31.415000,0 days 00:01:30.716000,0 days 00:01:30.213000,NaN,,NaN
10,GAS,Pierre Gasly,Alpine,5.0,0 days 00:01:31.462000,0 days 00:01:30.643000,0 days 00:01:30.216000,NaN,,NaN
4,NOR,Lando Norris,McLaren,6.0,0 days 00:01:31.107000,0 days 00:01:30.560000,0 days 00:01:30.267000,NaN,,NaN
1,VER,Max Verstappen,Red Bull Racing,7.0,0 days 00:01:31.303000,0 days 00:01:31.019000,0 days 00:01:30.423000,NaN,,NaN
55,SAI,Carlos Sainz,Williams,8.0,0 days 00:01:31.591000,0 days 00:01:30.844000,0 days 00:01:30.680000,NaN,,NaN
44,HAM,Lewis Hamilton,Ferrari,9.0,0 days 00:01:31.219000,0 days 00:01:31.009000,0 days 00:01:30.772000,NaN,,NaN
22,TSU,Yuki Tsunoda,Red Bull Racing,10.0,0 days 00:01:31.751000,0 days 00:01:31.228000,0 days 00:01:31.303000,NaN,,NaN


Results shape: (20, 22)


In [7]:
# inspect lap table 
laps_df = session.laps.copy()

lap_cols = [
    'Driver', 'DriverNumber', 'LapNumber', 'LapTime',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Compound', 'TyreLife', 'Stint',
    'IsPersonalBest', 'TrackStatus', 'PitOutTime', 'PitInTime'
]
existing_lap_cols = [c for c in lap_cols if c in laps_df.columns]

display(laps_df[existing_lap_cols].head(20))
print('Laps shape:', laps_df.shape)

,Driver,DriverNumber,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,Compound,TyreLife,Stint,IsPersonalBest,TrackStatus,PitOutTime,PitInTime
0,PIA,81,1.0,NaT,NaT,0 days 00:00:48.132000,0 days 00:00:29.724000,SOFT,1.0,1.0,False,1,0 days 00:17:48.888000,NaT
1,PIA,81,2.0,0 days 00:01:31.392000,0 days 00:00:29.287000,0 days 00:00:39.300000,0 days 00:00:22.805000,SOFT,2.0,1.0,True,1,NaT,NaT
2,PIA,81,3.0,0 days 00:02:01.543000,0 days 00:00:38.060000,0 days 00:00:50.877000,0 days 00:00:32.606000,SOFT,3.0,1.0,False,1,NaT,0 days 00:23:34.171000
3,PIA,81,4.0,NaT,NaT,0 days 00:00:53.751000,0 days 00:00:30.378000,SOFT,4.0,2.0,False,1,0 days 00:27:14.444000,NaT
4,PIA,81,5.0,0 days 00:01:59.252000,0 days 00:00:31.125000,0 days 00:00:52.778000,0 days 00:00:35.349000,SOFT,5.0,2.0,False,1,NaT,0 days 00:31:27.651000
5,PIA,81,6.0,NaT,NaT,NaT,NaT,SOFT,1.0,3.0,False,125,0 days 00:40:58.504000,0 days 00:43:35.790000
6,PIA,81,7.0,NaT,NaT,0 days 00:00:48.070000,0 days 00:00:31.594000,SOFT,1.0,4.0,False,1,0 days 00:50:38.265000,NaT
7,PIA,81,8.0,0 days 00:01:30.454000,0 days 00:00:28.883000,0 days 00:00:38.853000,0 days 00:00:22.718000,SOFT,2.0,4.0,True,1,NaT,NaT
8,PIA,81,9.0,0 days 00:02:02.039000,0 days 00:00:34.994000,0 days 00:00:50.643000,0 days 00:00:36.402000,SOFT,3.0,4.0,False,1,NaT,0 days 00:56:16.075000
9,PIA,81,10.0,NaT,NaT,0 days 00:00:51.711000,0 days 00:00:28.703000,SOFT,2.0,5.0,False,12,0 days 01:11:06.850000,NaT


Laps shape: (277, 31)


In [8]:
# save sample outputs so we have tangible artifacts from notebook 00
event_name_safe = str(session.event['EventName']).replace(' ', '_')
base_name = f"{season}_{event_name_safe}_{session_type}"

results_out = raw_dir / f"{base_name}_results.csv"
laps_out = raw_dir / f"{base_name}_laps.parquet"

results_df.to_csv(results_out, index=False)
laps_df.to_parquet(laps_out, index=False)

print('Saved:', results_out)
print('Saved:', laps_out)

Saved: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\2025_Bahrain_Grand_Prix_Q_results.csv
Saved: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\2025_Bahrain_Grand_Prix_Q_laps.parquet
